# Fetch CheXzero weights once

Downloads the 10 released CheXzero checkpoints from the authors' Google Drive folder (Tiu et al., 2022) and records their sizes and MD5 checksums. After a successful run, turn the output into a **private Kaggle dataset** so later notebooks never need Google Drive again.

Settings: Accelerator **None** (no GPU needed), Internet **On**. Run with **Save Version → Save & Run All (Commit)**.
If it fails with "Cannot retrieve the public link", Google Drive's daily download limit has been reached: wait a few hours (it resets within about a day) and save a new version.

In [ ]:
!pip install -q gdown
import gdown, glob, os, hashlib, json
OUT = "/kaggle/working/chexzero_weights"; os.makedirs(OUT, exist_ok=True)
gdown.download_folder("https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno",
                      output=OUT, quiet=False, remaining_ok=True)
pts = sorted(glob.glob(OUT + "/**/*.pt", recursive=True))
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
info = [{"file": os.path.relpath(p, OUT), "bytes": os.path.getsize(p), "md5": md5(p)} for p in pts]
json.dump(info, open(OUT + "/checksums.json", "w"), indent=1)
print(json.dumps(info, indent=1))
print("FOUND", len(pts), "checkpoints")
assert len(pts) == 10, "Not all 10 checkpoints downloaded"